In [ ]:
import os
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, UnidentifiedImageError
from pathlib import Path
from collections import Counter

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences



In [ ]:
DATA_DIR      = 'final_data'        
CSV_PATH      = 'final_data.csv'   
IMG_SIZE      = (224, 224)       
RANDOM_STATE  = 42
MAX_WORDS     = 1000             
MAX_SEQ_LEN   = 20               

CLASSES = ['glass', 'paper', 'metal', 'plastic']
NUM_CLASSES = len(CLASSES)

TEXT_TEMPLATES = {
    'glass'  : 'this is a glass waste item such as a broken bottle or glass jar found in garbage',
    'paper'  : 'this is a paper waste item such as newspaper or crumpled white paper discarded in bin',
    'metal'  : 'this is a metal waste item such as an aluminum can or tin can thrown away',
    'plastic': 'this is a plastic waste item such as a plastic bottle or packaging found in garbage',
}

print('Configuration loaded ')
print(f'Classes: {CLASSES}')
print(f'Image target size: {IMG_SIZE}')

In [ ]:
df = pd.read_csv(CSV_PATH)
print('Shape:', df.shape)
print('\nFirst 5 rows:')
display(df.head())

print('\nClass distribution:')
print(df['label'].value_counts())

# Bar chart
plt.figure(figsize=(7, 4))
df['label'].value_counts().plot(kind='bar', color=['#4C72B0','#DD8452','#55A868','#C44E52'])
plt.title('Number of Images per Class')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120)
plt.show()
print('Chart saved ')

In [ ]:
def is_valid_image(path: str) -> bool:
    """Return True if the file is a readable, non-corrupt image."""
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

before = len(df)
df['valid'] = df['image'].apply(is_valid_image)
df = df[df['valid']].drop(columns=['valid']).reset_index(drop=True)
after = len(df)

print(f'Images before cleaning : {before}')
print(f'Corrupted removed       : {before - after}')
print(f'Images after cleaning  : {after} ')

In [ ]:
print(df['image'].head())
import os
print(os.path.exists(df['image'].iloc[0]))

In [ ]:
def load_and_resize(path: str, size=IMG_SIZE) -> np.ndarray:
    """Load image RGB  resize to (224,224)  numpy array."""
    img = Image.open(path).convert('RGB').resize(size)
    return img_to_array(img)   # shape: (224, 224, 3), dtype float32

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, cls in zip(axes, CLASSES):
    sample_path = df[df['label'] == cls]['image'].iloc[0]
    arr = load_and_resize(sample_path).astype(np.uint8)
    ax.imshow(arr)
    ax.set_title(cls)
    ax.axis('off')
plt.suptitle('Sample Image per Class (224×224)', fontsize=13)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=120)
plt.show()
print('Sample images saved ')

In [ ]:
print('Loading images — this may take a few minutes...')

X_raw = []
valid_indices = []

for idx, row in df.iterrows():
    try:
        arr = load_and_resize(row['image'])
        X_raw.append(arr)
        valid_indices.append(idx)
    except Exception as e:
        print(f'  Skipped {row["image"]}: {e}')

df = df.loc[valid_indices].reset_index(drop=True)

X_images = np.array(X_raw, dtype='float32') / 255.0   # normalize to [0, 1]
print(f'X_images shape  : {X_images.shape}')
print(f'Pixel value range: [{X_images.min():.2f}, {X_images.max():.2f}]')
print('Images loaded and normalized ')

In [ ]:
# Integer encoding
le = LabelEncoder()
le.fit(CLASSES)
y_int = le.transform(df['label'].values)   # glass 0, metal 1, paper 2, plastic 3

# One-hot encoding
y_onehot = to_categorical(y_int, num_classes=NUM_CLASSES)

print('Label Encoder classes :', le.classes_)
print('y_int   shape         :', y_int.shape)
print('y_onehot shape        :', y_onehot.shape)
print('\nFirst 5 labels (int)  :', y_int[:5])
print('First 5 labels (onehot):')
print(y_onehot[:5])

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_images, y_onehot,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_int
)
y_temp_int = np.argmax(y_temp, axis=1)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp_int
)

total = len(X_images)
print(f'Total samples  : {total}')
print(f'Train  (70%)   : {len(X_train)}  ({len(X_train)/total*100:.1f}%)')
print(f'Val    (15%)   : {len(X_val)}   ({len(X_val)/total*100:.1f}%)')
print(f'Test   (15%)   : {len(X_test)}   ({len(X_test)/total*100:.1f}%)')
print('Split done ')

In [ ]:
# Augmentation
train_datagen = ImageDataGenerator(
    horizontal_flip=True,
    vertical_flip=False,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator()  

# Create generators
BATCH_SIZE = 32

train_generator = train_datagen.flow(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=RANDOM_STATE
)

val_generator = val_test_datagen.flow(
    X_val, y_val,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_generator = val_test_datagen.flow(
    X_test, y_test,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print('Augmentation generators created')

# Visualize augmented samples
sample_X = X_train[:1]          
sample_y = y_train[:1]
aug_gen  = train_datagen.flow(sample_X, sample_y, batch_size=1)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes[0, 0].imshow(sample_X[0])
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

for i, ax in enumerate(axes.flatten()[1:]):
    aug_img, _ = next(aug_gen)
    ax.imshow(np.clip(aug_img[0], 0, 1))
    ax.set_title(f'Aug {i+1}')
    ax.axis('off')

plt.suptitle('Data Augmentation Samples', fontsize=13)
plt.tight_layout()
plt.savefig('augmentation_samples.png', dpi=120)
plt.show()
print('Augmentation visualization saved')

In [ ]:
# Assign a text description to every image based on its class
df['text'] = df['label'].map(TEXT_TEMPLATES)

print('Sample descriptions:')
print(df[['label', 'text']].drop_duplicates().to_string(index=False))

In [ ]:
def clean_text(text: str) -> str:
    text = text.lower()                                      
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()                
    return text

df['text_clean'] = df['text'].apply(clean_text)

print('Before cleaning:', df['text'].iloc[0])
print('After  cleaning:', df['text_clean'].iloc[0])

In [ ]:
texts = df['text_clean'].tolist()

# Fit tokenizer on ALL texts (unique 4 descriptions, but covers full vocab)
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(texts)

word_index = tokenizer.word_index
print(f'Vocabulary size: {len(word_index)} words')
print('Sample word → index:', dict(list(word_index.items())[:10]))

# Convert to sequences
sequences = tokenizer.texts_to_sequences(texts)

print('\nSample text          :', texts[0])
print('Converted to sequence:', sequences[0])

In [ ]:
X_text = pad_sequences(
    sequences,
    maxlen=MAX_SEQ_LEN,
    padding='post',
    truncating='post'
)

print('X_text shape:', X_text.shape)   # (num_samples, MAX_SEQ_LEN)
print('\nFirst padded sequence:')
print(X_text[0])

In [ ]:
# Same label encoder as images
y_text_int    = le.transform(df['label'].values)
y_text_onehot = to_categorical(y_text_int, num_classes=NUM_CLASSES)

print('y_text_int    shape:', y_text_int.shape)
print('y_text_onehot shape:', y_text_onehot.shape)
print('\nSample (label  int  onehot):')
for i in range(4):
    print(f'  {df["label"].iloc[i]:8s}  {y_text_int[i]}  {y_text_onehot[i]}')

In [ ]:
X_text_train, X_text_temp, y_text_train, y_text_temp = train_test_split(
    X_text, y_text_onehot,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_text_int
)

y_text_temp_int = np.argmax(y_text_temp, axis=1)
X_text_val, X_text_test, y_text_val, y_text_test = train_test_split(
    X_text_temp, y_text_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_text_temp_int
)

total_t = len(X_text)
print(f'Text — Total  : {total_t}')
print(f'Text — Train  : {len(X_text_train)}  ({len(X_text_train)/total_t*100:.1f}%)')
print(f'Text — Val    : {len(X_text_val)}   ({len(X_text_val)/total_t*100:.1f}%)')
print(f'Text — Test   : {len(X_text_test)}   ({len(X_text_test)/total_t*100:.1f}%)')

In [ ]:
os.makedirs('processed_data', exist_ok=True)

np.save('processed_data/X_train.npy',    X_train)
np.save('processed_data/X_val.npy',      X_val)
np.save('processed_data/X_test.npy',     X_test)
np.save('processed_data/y_train.npy',    y_train)
np.save('processed_data/y_val.npy',      y_val)
np.save('processed_data/y_test.npy',     y_test)

np.save('processed_data/X_text_train.npy', X_text_train)
np.save('processed_data/X_text_val.npy',   X_text_val)
np.save('processed_data/X_text_test.npy',  X_text_test)
np.save('processed_data/y_text_train.npy', y_text_train)
np.save('processed_data/y_text_val.npy',   y_text_val)
np.save('processed_data/y_text_test.npy',  y_text_test)

import json
with open('processed_data/tokenizer_config.json', 'w') as f:
    json.dump(tokenizer.get_config(), f)

np.save('processed_data/label_classes.npy', le.classes_)

print('All processed data saved to processed_data/')
print('\nFiles saved:')
for f in sorted(os.listdir('processed_data')):
    path = os.path.join('processed_data', f)
    size_mb = os.path.getsize(path) / 1e6
    print(f'  {f:40s}  {size_mb:.1f} MB')